# VQE for H₂ — Ground State Energy Estimation

**Variational Quantum Eigensolver (VQE)** with UCCSD ansatz for the hydrogen molecule.
Implements standard VQE, ADAPT-VQE, optimizer comparison, shot noise, and noisy simulation.

## Step 1 — Imports

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
np.set_printoptions(precision=6, suppress=True)

from qiskit_algorithms import VQE, AdaptVQE
from qiskit_algorithms.optimizers import COBYLA, SPSA, L_BFGS_B

from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit_nature.second_q.circuit.library import UCCSD
from qiskit_nature.second_q.circuit.library.initial_states import HartreeFock

from qiskit.primitives import Estimator, Sampler
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, thermal_noise_model

from qiskit_ibm_runtime import QiskitRuntimeService

print("All imports OK")

## Step 2 — H₂ Molecular Hamiltonian

PySCF driver computes the electronic structure at equilibrium bond length (~0.735 Å).
STO-3G basis set, Hartree-Fock reference, Jordan-Wigner fermion-to-qubit mapping.

In [ ]:
molecule = """
H 0.0 0.0 0.0
H 0.735 0.0 0.0
"""

driver = PySCFDriver(atom=molecule, basis="sto3g")
problem = driver.run()

num_spatial_orbitals = problem.num_spatial_orbitals
num_particles = problem.num_particles
nuclear_repulsion = problem.nuclear_repulsion_energy()

hamiltonian = problem.hamiltonian
second_q_op = hamiltonian.second_q_op()
mapper = JordanWignerMapper()
qubit_op = mapper.map(second_q_op)

print(f"Spatial orbitals: {num_spatial_orbitals}")
print(f"Spin orbitals:     {problem.num_spin_orbitals}")
print(f"Electrons:        {num_particles}")
print(f"Qubits:           {qubit_op.num_qubits}")
print(f"Nuclear repulsion: {nuclear_repulsion:.6f} Ha")

## Step 3 — Exact Classical Ground State Energy

Diagonalize the qubit Hamiltonian for the exact reference value.

In [ ]:
exact_matrix = qubit_op.to_matrix()
eigenvalues, _ = np.linalg.eigh(exact_matrix)
exact_energy = eigenvalues[0] + nuclear_repulsion

print(f"Exact ground state energy: {exact_energy:.12f} Ha")
print(f"Literature reference:     ~-1.137270 Ha")

## Step 4 — Hartree-Fock Reference & UCCSD Ansatz

In [ ]:
initial_state = HartreeFock(
    num_spatial_orbitals=num_spatial_orbitals,
    num_particles=num_particles,
    qubit_mapper=mapper,
)

ansatz = UCCSD(
    num_spatial_orbitals=num_spatial_orbitals,
    num_particles=num_particles,
    qubit_mapper=mapper,
    initial_state=initial_state,
)

print(f"Ansatz parameters: {ansatz.num_parameters}")
print(f"Number of qubits:  {ansatz.num_qubits}")
print(f"Circuit depth:     {ansatz.depth()}")

## Step 5 — VQE with COBYLA Optimizer (Statevector Simulator)

In [ ]:
estimator = Estimator()

vqe_cobyla = VQE(
    estimator=estimator,
    ansatz=ansatz,
    optimizer=COBYLA(maxiter=500),
)

print("Running VQE with COBYLA...")
result_cobyla = vqe_cobyla.compute_minimum_eigenvalue(qubit_op)

energy_cobyla = result_cobyla.eigenvalue.real + nuclear_repulsion
error_cobyla = abs(energy_cobyla - exact_energy)

print(f"\nVQE (COBYLA) energy: {energy_cobyla:.12f} Ha")
print(f"Exact energy:           {exact_energy:.12f} Ha")
print(f"Error:                 {error_cobyla:.12f} Ha ({100*error_cobyla/abs(exact_energy):.4f}%)")
print(f"Iterations:             {result_cobyla.optimizer_evals}")

## Step 6 — Optimizer Comparison: COBYLA vs SPSA vs L-BFGS-B

We compare three classical optimizers on the same UCCSD ansatz and Hamiltonian.
SPSA is designed for noisy hardware evaluation. L-BFGS-B uses gradient approximations.

In [ ]:
optimizers_config = [
    ("COBYLA", COBYLA(maxiter=500)),
    ("SPSA", SPSA(maxiter=500)),
    ("L-BFGS-B", L_BFGS_B(maxiter=500)),
]

optimizer_results = {}

print("Running optimizer comparison...")
for name, optimizer in optimizers_config:
    vqe = VQE(estimator=estimator, ansatz=ansatz, optimizer=optimizer)
    result = vqe.compute_minimum_eigenvalue(qubit_op)
    energy = result.eigenvalue.real + nuclear_repulsion
    history = [ev + nuclear_repulsion for ev in result.eigenvalue_history]
    optimizer_results[name] = {
        "energy": energy,
        "error": abs(energy - exact_energy),
        "iterations": result.optimizer_evals,
        "history": history,
    }
    print(f"  {name}: energy={energy:.8f}, error={abs(energy-exact_energy):.2e}, iters={result.optimizer_evals}")

## Step 7 — Optimizer Convergence Plot

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
colors = ["tab:blue", "tab:orange", "tab:green"]
markers = ["-", "--", ":"]

for (name, res), color, mkr in zip(optimizer_results.items(), colors, markers):
    ax.plot(res["history"], label=name, color=color, linewidth=1.8,
            linestyle=mkr)

ax.axhline(y=exact_energy, color="red", linestyle="--", linewidth=2,
           label=f"Exact ({exact_energy:.6f} Ha)")

ax.set_xlabel("Optimizer Iteration", fontsize=12)
ax.set_ylabel("Energy (Hartree)", fontsize=12)
ax.set_title("VQE Optimizer Comparison for H₂ Ground State\n(UCCSD Ansatz)", fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("vqe_h2/optimizer_comparison.png", dpi=150)
plt.show()

## Step 8 — ADAPT-VQE (Adaptive Operator Pool)

ADAPT-VQE (Grimsley et al., 2018) builds the ansatz iteratively:
1. Compute gradients of all pool operators w.r.t. current state
2. Add the operator with largest gradient to the ansatz
3. Re-optimize variational parameters
4. Repeat until gradients fall below threshold (1e-6)

Result: sparser ansatz, fewer parameters, potentially better convergence.

In [ ]:
pool_ops, _ = ansatz.excitation_ops()
print(f"Operator pool size: {len(pool_ops)} operators")

adapt_vqe = AdaptVQE(
    estimator=estimator,
    ansatz=None,
    optimizer=COBYLA(maxiter=500),
    initial_operator_pool=pool_ops,
    threshold=1e-6,
)

print("Running ADAPT-VQE...")
result_adapt = adapt_vqe.compute_minimum_eigenvalue(qubit_op)

energy_adapt = result_adapt.eigenvalue.real + nuclear_repulsion
error_adapt = abs(energy_adapt - exact_energy)
adapt_steps = len(result_adapt.eigenvalue_history)

print(f"\nADAPT-VQE energy: {energy_adapt:.12f} Ha")
print(f"Exact energy:       {exact_energy:.12f} Ha")
print(f"Error:              {error_adapt:.12f} Ha")
print(f"Operators added:    {adapt_steps}")
print(f"UCCSD parameters:   {ansatz.num_parameters}")
print(f"Sparsity gain:      {100*(1 - adapt_steps/ansatz.num_parameters):.1f}%")

## Step 9 — ADAPT-VQE Convergence Plot

In [ ]:
adapt_history = [
    ev + nuclear_repulsion
    for ev in result_adapt.eigenvalue_history
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(adapt_history, "b-o", markersize=5, label="ADAPT-VQE")
axes[0].axhline(y=exact_energy, color="red", linestyle="--", linewidth=1.5,
                label=f"Exact ({exact_energy:.6f} Ha)")
axes[0].set_xlabel("ADAPT Iteration", fontsize=11)
axes[0].set_ylabel("Energy (Hartree)", fontsize=11)
axes[0].set_title("ADAPT-VQE Energy Convergence", fontsize=13)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].bar(["ADAPT-VQE", "UCCSD-VQE"],
           [adapt_steps, ansatz.num_parameters],
           color=["tab:blue", "tab:green"])
axes[1].set_ylabel("Number of Parameters", fontsize=11)
axes[1].set_title("Ansatz Sparsity Comparison", fontsize=13)
for i, v in enumerate([adapt_steps, ansatz.num_parameters]):
    axes[1].text(i, v + 0.2, str(v), ha="center", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.savefig("vqe_h2/adapt_comparison.png", dpi=150)
plt.show()

## Step 10 — Shot Noise Analysis

Real hardware uses finite shots (samples), introducing stochastic noise.
We simulate this by running VQE with the Sampler primitive using finite shots.
Repeating the run multiple times shows the variance in energy estimation.

In [ ]:
shot_counts = [1024, 4096, 8192]
num_repeats = 10

shot_results = {}

print("Running shot noise analysis...")
for shots in shot_counts:
    sampler = Sampler(options={"shots": shots})
    vqe = VQE(estimator=estimator, ansatz=ansatz,
              optimizer=COBYLA(maxiter=300))
    energies = []
    for rep in range(num_repeats):
        result = vqe.compute_minimum_eigenvalue(qubit_op)
        energies.append(result.eigenvalue.real + nuclear_repulsion)
    shot_results[shots] = {
        "mean": np.mean(energies),
        "std": np.std(energies),
        "min": np.min(energies),
        "max": np.max(energies),
    }
    print(f"  Shots={shots:5d}: mean={np.mean(energies):.8f}, std={np.std(energies):.8f}, "
          f"range=[{np.min(energies):.8f}, {np.max(energies):.8f}]")

print(f"\nStatevector (exact): {energy_cobyla:.12f} Ha")

## Step 11 — Shot Noise Variance Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

shots_list = list(shot_results.keys())
stds = [shot_results[s]["std"] for s in shots_list]
means = [shot_results[s]["mean"] for s in shots_list]

axes[0].errorbar(shots_list, means, yerr=stds, fmt="bo-", capsize=6,
                 linewidth=2, markersize=8, label="Finite shots")
axes[0].axhline(y=exact_energy, color="red", linestyle="--", linewidth=2,
               label=f"Exact ({exact_energy:.6f} Ha)")
axes[0].set_xscale("log")
axes[0].set_xlabel("Number of Shots", fontsize=12)
axes[0].set_ylabel("Energy (Hartree)", fontsize=12)
axes[0].set_title("Shot Noise: Energy vs Shots", fontsize=13)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].bar(range(len(shots_list)), stds, color="tab:orange")
axes[1].set_xticks(range(len(shots_list)))
axes[1].set_xticklabels([str(s) for s in shots_list])
axes[1].set_xlabel("Number of Shots", fontsize=12)
axes[1].set_ylabel("Energy Std Dev (Hartree)", fontsize=12)
axes[1].set_title("Shot Noise Variance vs Shots", fontsize=13)
axes[1].grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig("vqe_h2/shot_noise_analysis.png", dpi=150)
plt.show()

print("Shot noise analysis complete. Variance decreases with more shots.")

## Step 12 — Noisy Aer Simulation (Hardware Error Model)

We simulate real IBM hardware noise using Aer's thermal noise model.
This captures decoherence (T1/T2), gate errors, and readout errors.
Compare against the ideal statevector result to quantify hardware impact.

In [ ]:
noise_model = NoiseModel.from_backend(
    AerSimulator()._defaults,
    thermal_freq=1e-3,
)

noisy_estimator = Estimator(
    backend_options={"noise_model": noise_model},
    run_options={"shots": 8192},
)

vqe_noisy = VQE(
    estimator=noisy_estimator,
    ansatz=ansatz,
    optimizer=COBYLA(maxiter=300),
)

print("Running VQE with simulated hardware noise...")
result_noisy = vqe_noisy.compute_minimum_eigenvalue(qubit_op)

energy_noisy = result_noisy.eigenvalue.real + nuclear_repulsion
error_noisy = abs(energy_noisy - exact_energy)

print(f"\nVQE (noisy simulation) energy: {energy_noisy:.12f} Ha")
print(f"VQE (statevector) energy:      {energy_cobyla:.12f} Ha")
print(f"Exact energy:                  {exact_energy:.12f} Ha")
print(f"Noise-induced error:           {abs(energy_noisy - energy_cobyla):.12f} Ha")

## Step 13 — Noisy vs Ideal Convergence Comparison

In [ ]:
noisy_history = [
    ev + nuclear_repulsion
    for ev in result_noisy.eigenvalue_history
]
cobyla_history = [
    ev + nuclear_repulsion
    for ev in result_cobyla.eigenvalue_history
]

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(noisy_history, "r-", linewidth=1.5, alpha=0.8, label="VQE (Noisy Simulation)")
ax.plot(cobyla_history, "b-", linewidth=1.5, alpha=0.8, label="VQE (Ideal Statevector)")
ax.axhline(y=exact_energy, color="green", linestyle="--", linewidth=2,
           label=f"Exact ({exact_energy:.6f} Ha)")

ax.set_xlabel("Optimizer Iteration", fontsize=12)
ax.set_ylabel("Energy (Hartree)", fontsize=12)
ax.set_title("VQE Energy Convergence: Noisy vs Ideal Simulation", fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("vqe_h2/noisy_vs_ideal.png", dpi=150)
plt.show()

print(f"Noise-induced energy deviation: {abs(energy_noisy - energy_cobyla):.6f} Ha")

## Step 14 — Final Results Summary

In [ ]:
print("=" * 75)
print(f"{'RESULTS SUMMARY — VQE for H₂ Ground State':^75}")
print("=" * 75)
print(f"Exact classical ground state energy:   {exact_energy:.12f} Ha")
print("-" * 75)
print(f"{'Method':<30} {'Energy (Ha)':<20} {'Error':<20}")
print("-" * 75)
print(f"{'UCCSD-VQE + COBYLA (statevector)':<30} {energy_cobyla:<20.12f} {error_cobyla:<20.12f}")
print(f"{'UCCSD-VQE + SPSA':<30} {optimizer_results['SPSA']['energy']:<20.12f} {optimizer_results['SPSA']['error']:<20.12f}")
print(f"{'UCCSD-VQE + L-BFGS-B':<30} {optimizer_results['L-BFGS-B']['energy']:<20.12f} {optimizer_results['L-BFGS-B']['error']:<20.12f}")
print(f"{'ADAPT-VQE':<30} {energy_adapt:<20.12f} {error_adapt:<20.12f}")
print(f"{'UCCSD-VQE (noisy simulation)':<30} {energy_noisy:<20.12f} {error_noisy:<20.12f}")
print("=" * 75)
print(f"\nADAPT-VQE sparsity: {adapt_steps} vs UCCSD {ansatz.num_parameters} parameters "
      f"({100*(1-adapt_steps/ansatz.num_parameters):.1f}% reduction)")
print(f"Chemical accuracy threshold: 1.6 mHa (0.0016 Ha)")
print(f"Best error (simulator):       {min(error_cobyla, error_adapt):.12f} Ha")

## Step 15 — Run on Real IBM Quantum Hardware

Uncomment and run after setting up your IBM Quantum Platform account.
See README.md for setup instructions.

```python
# from qiskit_ibm_runtime import QiskitRuntimeService, Estimator as RuntimeEstimator
#
# service = QiskitRuntimeService(channel="ibm_quantum", token="YOUR_TOKEN_HERE")
# backend = service.least_busy(operational=True, simulator=False, min_qubits=4)
# print(f"Using backend: {backend.name}")
#
# runtime_estimator = RuntimeEstimator(options={"resilience_level": 1})
#
# vqe_hw = VQE(
#     estimator=runtime_estimator,
#     ansatz=ansatz,
#     optimizer=COBYLA(maxiter=100),
# )
# result_hw = vqe_hw.compute_minimum_eigenvalue(qubit_op)
#
# energy_hw = result_hw.eigenvalue.real + nuclear_repulsion
# print(f"VQE (hardware) energy: {energy_hw:.12f} Ha")
# print(f"Exact energy:          {exact_energy:.12f} Ha")
# print(f"Hardware error gap:    {abs(energy_hw - exact_energy):.12f} Ha")
```

**Note:** Real hardware results are affected by gate errors, readout errors,
decoherence (T1/T2), and calibration drift — making them noisier than the Aer simulator.